# Análise de Preços de Combustíveis no Brasil
Dados da ANP (2025-2026), tratados em Python e armazenados em SQLite.
Objetivo: entender variação de preço por estado, produto e período, com destaque para Minas Gerais.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
engine = create_engine("sqlite:///../dados/tratados/combustiveis.db")

## Preço médio por estado e produto
Qual o combustível mais caro/barato em cada estado do Brasil?

In [14]:
with open("../sql/preco_medio_UF.sql") as f:
    query1 = f.read()

resultado = pd.read_sql(query1, engine)
resultado.head(20)

,Estado - Sigla,Produto,preco_medio
0,AC,DIESEL S10,7.875376
1,AC,DIESEL,7.785116
2,AC,GASOLINA ADITIVADA,7.660176
3,AC,GASOLINA,7.613423
4,AC,ETANOL,5.452142
5,AL,DIESEL,6.828674
6,AL,GASOLINA ADITIVADA,6.636435
7,AL,DIESEL S10,6.480869
8,AL,GASOLINA,6.462194
9,AL,ETANOL,5.047607


## Preço médio de gasolina por estado

Os estados do Norte lideram os maiores preços — Acre (R$ 7,61), Amazonas (R$ 7,25) 
e Roraima (R$ 7,10) — provavelmente refletindo o custo logístico de transporte de 
combustível para regiões mais distantes dos polos de refino.

Minas Gerais aparece com preço médio de R$ 6,21, mais barato que a maioria dos 
estados listados até aqui.

In [3]:
resultado[resultado["Produto"] == "GASOLINA"].sort_values("preco_medio", ascending=False)

,Estado - Sigla,Produto,preco_medio
3,AC,GASOLINA,7.613423
12,AM,GASOLINA,7.252951
120,RR,GASOLINA,7.097796
115,RO,GASOLINA,7.011182
137,SE,GASOLINA,6.702515
149,TO,GASOLINA,6.644901
24,BA,GASOLINA,6.624180
86,PE,GASOLINA,6.518134
97,PR,GASOLINA,6.506002
30,CE,GASOLINA,6.502803


## Evolução mensal do preço da gasolina (jan/2025 a jun/2026)
**Observação:** o preço partiu de R$ 6,18 em janeiro/2025, teve uma leve alta em 
fevereiro e depois caiu gradualmente até um mínimo de R$ 6,18 em agosto/2025 — 
praticamente voltando ao patamar inicial. A partir de 2026, porém, o cenário muda: 
o preço sobe de forma consistente, saindo de R$ 6,33 em janeiro para um pico de 
R$ 6,76 em abril/2026, a maior alta acumulada de toda a série.

In [4]:
with open("../sql/variacao_mensal.sql") as f:
    query2 = f.read()
resultado = pd.read_sql(query2, engine)
resultado

,ano_mes,preco_medio
0,2025-01,6.183839
1,2025-02,6.367177
2,2025-03,6.349952
3,2025-04,6.316235
4,2025-05,6.283602
5,2025-06,6.226324
6,2025-07,6.211219
7,2025-08,6.182066
8,2025-09,6.191606
9,2025-10,6.215174


## Variação percentual mensal do preço da gasolina

Usando LAG() para comparar cada mês com o anterior, fica claro que a alta de 
preços em 2026 não foi gradual: os maiores saltos da série completa aconteceram 
em março (+4,61%) e abril de 2026 (+2,41%), concentrando boa parte do aumento 
visto no gráfico anterior em apenas dois meses.

Já 2025 teve variações mais amenas, entre -0,91% e +2,96%, sem nenhum mês de 
alta ou queda tão acentuada quanto os de 2026.

(Janeiro/2025 aparece sem variação por ser o primeiro mês da série, sem mês 
anterior para comparação.)

In [5]:
with open("../sql/variacao_percentual_mes.sql") as f:
    query3 = f.read()
resultado = pd.read_sql(query3, engine)
resultado

,ano_mes,preco_medio,preco_mes_anterior,variacao_percentual
0,2025-01,6.183839,NaN,NaN
1,2025-02,6.367177,6.183839,2.96
2,2025-03,6.349952,6.367177,-0.27
3,2025-04,6.316235,6.349952,-0.53
4,2025-05,6.283602,6.316235,-0.52
5,2025-06,6.226324,6.283602,-0.91
6,2025-07,6.211219,6.226324,-0.24
7,2025-08,6.182066,6.211219,-0.47
8,2025-09,6.191606,6.182066,0.15
9,2025-10,6.215174,6.191606,0.38


## Top 10 municípios com gasolina mais cara

**Observação:** Parintins (AM) lidera como o município com a gasolina mais cara 
do Brasil, a R$ 8,46/litro, seguido por Cruzeiro do Sul (AC) e Rio Branco (AC), 
ambos acima de R$ 7,40. O ranking é dominado pela região Norte (AM, AC, RO, RR), 
com algumas exceções pontuais no Nordeste (BA, AL) e Sudeste (SP - Barueri).

In [8]:
with open("../sql/ranking_caras.sql") as f:
    query4 = f.read()
resultado = pd.read_sql(query4, engine)
resultado

,Municipio,Estado - Sigla,preco_medio
0,PARINTINS,AM,8.455468
1,CRUZEIRO DO SUL,AC,8.029055
2,RIO BRANCO,AC,7.429032
3,PORTO SEGURO,BA,7.379497
4,BAGE,RS,7.231848
5,DELMIRO GOUVEIA,AL,7.172629
6,PIMENTA BUENO,RO,7.156604
7,LIVRAMENTO DE NOSSA SENHORA,BA,7.121778
8,BARUERI,SP,7.118942
9,BOA VISTA,RR,7.097796


## Top 10 municípios com gasolina mais barata

**Observação:** Goiatuba (GO) tem a gasolina mais barata do país, a R$ 5,81/litro. 
A diferença entre o município mais caro (Parintins, R$ 8,46) e o mais barato 
(Goiatuba, R$ 5,81) é de R$ 2,65 — ou seja, o litro de gasolina pode custar 
quase 46% a mais dependendo só da localização. O Maranhão aparece 3 vezes 
nesse ranking dos mais baratos.

In [10]:
with open("../sql/ranking_baratas.sql") as f:
    query4b = f.read()
resultado = pd.read_sql(query4b, engine)
resultado

,Municipio,Estado - Sigla,preco_medio
0,GOIATUBA,GO,5.813510
1,SAO JOSE DE RIBAMAR,MA,5.823886
2,TAUBATE,SP,5.845451
3,SAO LUIS,MA,5.845519
4,ANANINDEUA,PA,5.867924
5,JACAREI,SP,5.877549
6,CAMPO GRANDE,MS,5.945919
7,SAO JOSE DOS CAMPOS,SP,5.961389
8,AMARANTE DO MARANHAO,MA,5.990000
9,GAVIAO,BA,5.990000


## Minas Gerais vs. Resto do Brasil

**Observação:** em todos os combustíveis líquidos (Diesel, Diesel S10, Etanol, 
Gasolina e Gasolina Aditivada), Minas Gerais fica consistentemente mais barato 
que a média do resto do Brasil — a maior diferença é no Etanol (R$ 4,42 vs 
R$ 4,56, MG mais barato em ~3%).

A única exceção é o GNV: em Minas Gerais o preço médio é R$ 5,03, cerca de 
8% mais caro que a média do resto do país (R$ 4,66) — possivelmente refletindo 
menor infraestrutura ou rede de distribuição de gás veicular no estado 
comparado a outras regiões.

In [11]:
with open("../sql/minas_vs_media_geral.sql") as f:
    query5 = f.read()
resultado = pd.read_sql(query5, engine)
resultado

,grupo,Produto,preco_medio
0,Minas Gerais,DIESEL,6.232494
1,Resto do Brasil,DIESEL,6.382712
2,Minas Gerais,DIESEL S10,6.358084
3,Resto do Brasil,DIESEL S10,6.452488
4,Minas Gerais,ETANOL,4.421282
5,Resto do Brasil,ETANOL,4.560107
6,Minas Gerais,GASOLINA,6.207948
7,Resto do Brasil,GASOLINA,6.370316
8,Minas Gerais,GASOLINA ADITIVADA,6.426470
9,Resto do Brasil,GASOLINA ADITIVADA,6.564325
